# TernakKu — Modul 1: Estimasi Bobot Sapi (Reproducible di Colab)

Notebook ini **menarik kode + data dari GitHub** lalu melatih model estimasi bobot dari ukuran tubuh (lingkar dada, panjang badan, tinggi gumba).

**Reproducible** karena: versi paket dikunci (`requirements.txt`), seed tetap (`42`), dan data ikut di repo. Run di Colab mana pun -> angka identik.

Repo: https://github.com/skivanoclaire/ternakku

Untuk presentasi: klik **Runtime > Run all**, audiens melihat kode dan hasilnya keluar langsung.

## 1. Tarik kode + data dari GitHub & pasang paket (versi terkunci)

In [ ]:
!git clone https://github.com/skivanoclaire/ternakku.git
%cd ternakku
!pip install -q -r requirements.txt
print('Siap. Versi terkunci terpasang.')

## 2. Bersihkan dataset publik (Hereford + Horqin)
Menghasilkan `data/out/pengukuran_public.csv` dengan kolom skema seragam.

In [ ]:
!python prep_data.py --indir data/raw --outdir data/out

## 3. Latih & bandingkan metode (seed=42 -> hasil identik)
Linear vs power-law (log-log) vs Random Forest vs XGBoost, dengan evaluasi 5-fold acak + lintas-dataset.

In [ ]:
!python train_modul1.py --csv data/out/pengukuran_public.csv --outdir data/out

## 4. Lihat metrik & contoh prediksi

In [ ]:
import json, joblib, numpy as np
print(json.dumps(json.load(open('data/out/metrics.json')), indent=2, ensure_ascii=False))

b = joblib.load('data/out/model_modul1.joblib')
ld, pb, tg = 195.0, 165.0, 128.0
if b['kind'] == 'loglog':
    kg = float(np.exp(b['a'] + b['b']*np.log(ld)))
else:
    kg = float(b['model'].predict([[ld, pb, tg]])[0])
print(f"\nContoh: LD={ld} PB={pb} gumba={tg} -> estimasi {kg:.1f} kg ({b['model_ver']})")

## 5. (Opsional) Visualisasi hubungan lingkar dada vs bobot

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('data/out/pengukuran_public.csv')
for name, grp in df.groupby('src_dataset'):
    plt.scatter(grp['lingkar_dada_cm'], grp['bobot_timbang_kg'], s=8, label=name, alpha=0.5)
plt.xlabel('Lingkar dada (cm)'); plt.ylabel('Bobot timbang (kg)')
plt.legend(); plt.title('Lingkar dada vs bobot'); plt.show()